# VQA-RAD Pilot — MedGemma-4B-it (Zero-Shot Closed-Ended Medical VQA)

**Model:** google/medgemma-4b-it (zero-shot, no fine-tuning, no few-shot examples)
**Sample:** same 100 closed-ended QA pairs as the Qwen2.5-VL run — identical seed, same sampling code,
so this reproduces the exact same 100 pairs for a fair head-to-head comparison.
**Requires:** Hugging Face access to google/medgemma-4b-it (accept the license) and an HF_TOKEN.

Run all cells top to bottom on a Colab or Kaggle GPU runtime (T4 or better).

In [ ]:
!pip install -q transformers accelerate pillow pandas scikit-learn torch datasets

## Step 0 — Hugging Face login

Make sure you've accepted the license at huggingface.co/google/medgemma-4b-it first.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN").strip()
login(hf_token)

## Step 1 — Load VQA-RAD and filter to the closed-ended (yes/no) subset

Same as the Qwen run.

In [ ]:
from datasets import load_dataset
import pandas as pd

vqa_rad = load_dataset("flaviagiammarino/vqa-rad")
full_df = pd.concat(
    [vqa_rad["train"].to_pandas(), vqa_rad["test"].to_pandas()], ignore_index=True
)

full_df["answer_norm"] = full_df["answer"].astype(str).str.strip().str.lower()
closed_df = full_df[full_df["answer_norm"].isin(["yes", "no"])].reset_index(drop=True)
print(f"Closed-ended (yes/no) pairs available: {len(closed_df)}")
print(closed_df["answer_norm"].value_counts())

README.md:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

data/train-00000-of-00001-eb8844602202be(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-eb8844602202be(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-e5bc3d208bb4dee(…): reconstructing file:   0%|          |  0.00B / 10.3MB            

data/test-00000-of-00001-e5bc3d208bb4dee(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1793 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/451 [00:00<?, ? examples/s]

Closed-ended (yes/no) pairs available: 1191
answer_norm
no     606
yes    585
Name: count, dtype: int64


## Step 2 — Same sample as before (SAME seed = same 100 pairs as the Qwen run)

In [ ]:
SEED = 42
N_TOTAL = 100
N_PER_ANSWER = N_TOTAL // 2

sampled = (
    closed_df.groupby("answer_norm", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_PER_ANSWER, len(g)), random_state=SEED))
    .reset_index(drop=True)
)
print(f"Total sampled: {len(sampled)}")
sampled[["question", "answer_norm"]].to_csv("vqa_rad_pilot_sample_medgemma.csv", index=False)
sampled[["question", "answer_norm"]].head(10)

Total sampled: 100


/tmp/ipykernel_816/778265515.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(N_PER_ANSWER, len(g)), random_state=SEED))


,question,answer_norm
0,is this a normal image?,no
1,is there evidence of herniation of the small b...,no
2,was the patient positioned inappropriately?,no
3,does the left temporal lobe appear normal?,no
4,is this an anterior-posterior image,no
5,are the ventricles an abnormal size?,no
6,is a pleural effusion present?,no
7,are the lungs increased in size?,no
8,is the cardiac silhouette enlarged?,no
9,is there cardiomegaly?,no


## Step 3 — Save the sampled images to disk (same decode fix as before)

In [ ]:
from pathlib import Path
from PIL import Image
import io

IMG_DIR = Path("vqa_rad_pilot_images_medgemma")
IMG_DIR.mkdir(exist_ok=True)

def to_pil(img_field):
    if isinstance(img_field, dict):
        return Image.open(io.BytesIO(img_field["bytes"]))
    return img_field

image_paths = []
for i, row in sampled.iterrows():
    path = IMG_DIR / f"{i}.jpg"
    to_pil(row["image"]).convert("RGB").save(path)
    image_paths.append(str(path))

sampled["image_path"] = image_paths

## Step 4 — Load MedGemma-4B-it

MedGemma uses the Gemma 3 multimodal architecture, so it loads via `AutoModelForImageTextToText`
(different class than Qwen2.5-VL) and uses `apply_chat_template` directly for both text and image.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

## Step 5 — Zero-shot prompt and inference loop

Same prompt wording as the Qwen run, so any difference in results reflects the model, not the prompt.

In [ ]:
def build_prompt(question):
    return (
        "You are a radiology assistant. Look at this medical image and answer the following "
        "question with ONLY the single word 'Yes' or 'No'.\n\n"
        f"Question: {question}"
    )

def predict(image_path, question):
    image = Image.open(image_path).convert("RGB")
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_prompt(question)},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        generation = model.generate(**inputs, max_new_tokens=16, do_sample=False)
        generation = generation[0][input_len:]
    return processor.decode(generation, skip_special_tokens=True).strip()

def parse_yes_no(raw_output):
    lowered = raw_output.lower()
    idx_yes = lowered.find("yes")
    idx_no = lowered.find("no")
    if idx_yes == -1 and idx_no == -1:
        return "unparsed"
    if idx_yes == -1:
        return "no"
    if idx_no == -1:
        return "yes"
    return "yes" if idx_yes < idx_no else "no"


In [ ]:
results = []
for i, row in sampled.iterrows():
    raw = predict(row["image_path"], row["question"])
    pred = parse_yes_no(raw)
    results.append({
        "question": row["question"],
        "true_answer": row["answer_norm"],
        "predicted_answer": pred,
        "raw_output": raw,
    })
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(sampled)} done")

results_df = pd.DataFrame(results)
results_df.to_csv("vqa_rad_pilot_results_medgemma.csv", index=False)
results_df.head()

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done


,question,true_answer,predicted_answer,raw_output
0,is this a normal image?,no,no,No
1,is there evidence of herniation of the small b...,no,no,No
2,was the patient positioned inappropriately?,no,no,No
3,does the left temporal lobe appear normal?,no,no,No
4,is this an anterior-posterior image,no,yes,Yes


## Step 6 — Score the results

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

valid = results_df[results_df["predicted_answer"] != "unparsed"]
print(f"Parsed cleanly: {len(valid)}/{len(results_df)}")

acc = accuracy_score(valid["true_answer"], valid["predicted_answer"])
print(f"Overall closed-ended accuracy: {acc:.3f}\n")

print(classification_report(valid["true_answer"], valid["predicted_answer"], zero_division=0))

cm = confusion_matrix(valid["true_answer"], valid["predicted_answer"], labels=["yes", "no"])
cm_df = pd.DataFrame(cm, index=["true_yes", "true_no"], columns=["pred_yes", "pred_no"])
cm_df.to_csv("vqa_rad_pilot_confusion_matrix_medgemma.csv")
cm_df

Parsed cleanly: 100/100
Overall closed-ended accuracy: 0.820

              precision    recall  f1-score   support

          no       0.85      0.78      0.81        50
         yes       0.80      0.86      0.83        50

    accuracy                           0.82       100
   macro avg       0.82      0.82      0.82       100
weighted avg       0.82      0.82      0.82       100



,pred_yes,pred_no
true_yes,43,7
true_no,11,39


## Step 7 — Misclassified cases

In [ ]:
wrong = results_df[results_df["predicted_answer"] != results_df["true_answer"]]
print(f"{len(wrong)} misclassified out of {len(results_df)}")
wrong[["question", "true_answer", "predicted_answer", "raw_output"]]

18 misclassified out of 100


,question,true_answer,predicted_answer,raw_output
4,is this an anterior-posterior image,no,yes,Yes
11,are the hepatic lesions ring enhancing?,no,yes,Yes
19,is the fat surrounding the pancreas normal?,no,yes,Yes
20,are there any pulmonary findings?,no,yes,Yes
24,is a cystic cavity present in the left kidney ...,no,yes,Yes
36,is there more than one intussusception present?,no,yes,Yes
41,are the ventricles normal?,no,yes,Yes
42,is there restricted diffusion?,no,yes,Yes
45,is there a fracture in the vertebrae seen?,no,yes,Yes
47,is this a saggital view of the brain?,no,yes,Yes
